# Module 4 — Le RAG : récupérer + générer

## Objectif
Injecter le contexte récupéré (Qdrant) dans le prompt, et mesurer **SANS vs AVEC** RAG.

### L'argument massue
Même question **sans** contexte (le modèle invente) puis **avec** contexte (réponse juste et sourcée). C'est le moment « déclic ».

In [ ]:
#r "nuget: Microsoft.SemanticKernel, 1.74.0"
#r "nuget: Microsoft.SemanticKernel.Connectors.Ollama, 1.74.0-alpha"
#r "nuget: Microsoft.SemanticKernel.Connectors.Qdrant, 1.74.0-preview"
#pragma warning disable SKEXP0070, SKEXP0001
using Microsoft.Extensions.AI;            // IEmbeddingGenerator (moderne)
using Microsoft.Extensions.VectorData;    // attributs VectorStore + collection
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.Connectors.Qdrant;
using Qdrant.Client;
using System.Security.Cryptography; using System.Text;
var endpoint = new Uri("http://localhost:11434");
// Embeddings MODERNES : IEmbeddingGenerator (remplace ITextEmbeddingGenerationService)
var embed = Kernel.CreateBuilder().AddOllamaEmbeddingGenerator("nomic-embed-text", endpoint)
    .Build().GetRequiredService<IEmbeddingGenerator<string, Embedding<float>>>();
var store = new QdrantVectorStore(new QdrantClient("localhost"), ownsClient: true);
ulong Id(string s) => BitConverter.ToUInt64(MD5.HashData(Encoding.UTF8.GetBytes(s)), 0);

In [ ]:
// Un enregistrement Qdrant : clé + texte + vecteur (768 = dimension de nomic-embed-text)
public sealed class Doc {
    [VectorStoreKey] public ulong Id { get; set; }
    [VectorStoreData] public string Text { get; set; } = "";
    [VectorStoreData(IsIndexed = true)] public string Source { get; set; } = "";
    [VectorStoreVector(768, DistanceFunction = DistanceFunction.CosineSimilarity)]
    public ReadOnlyMemory<float> Vector { get; set; }
}

In [ ]:
var kernel = Kernel.CreateBuilder().AddOllamaChatCompletion("gemma3:4b", endpoint).Build();
var collection = store.GetCollection<ulong, Doc>("demo_kb");
await collection.EnsureCollectionExistsAsync();

async Task Add(string t, string s) {
    var v = (await embed.GenerateAsync(t)).Vector;
    await collection.UpsertAsync(new Doc { Id = Id(t), Text = t, Source = s, Vector = v });
}
await Add("Wi-Fi instable : redemarrer le point d'acces, verifier le canal 5GHz.", "reseau");
await Add("Mot de passe oublie : self-service de reset.", "compte");

async Task<string> Retrieve(string q, int k = 3) {
    var qv = (await embed.GenerateAsync(q)).Vector;
    var passages = new List<string>();
    await foreach (var r in collection.SearchAsync(qv, top: k)) passages.Add(r.Record.Text);
    return passages.Count == 0 ? "(aucun contexte)" : string.Join("\n", passages);
}

In [ ]:
var question = "Que faire si le wifi est instable ?";

var sans = await kernel.InvokePromptAsync("{{$q}}", new KernelArguments { ["q"] = question });
Console.WriteLine("--- SANS RAG ---\n" + sans + "\n");

var ctx = await Retrieve(question);
var avec = await kernel.InvokePromptAsync(
    "Réponds UNIQUEMENT à partir du contexte, sinon dis que tu ne sais pas.\nContexte:\n{{$c}}\n\nQuestion: {{$q}}",
    new KernelArguments { ["c"] = ctx, ["q"] = question });
Console.WriteLine("--- AVEC RAG ---\n" + avec);

**À observer** : avec RAG, la réponse colle à la procédure interne et cesse d'inventer.

## Récap
- RAG = `Retrieve(contexte)` (Qdrant) → prompt → génération.
- Consigne d'honnêteté + fallback obligatoires.